# NLP Lab Assignment 10 – Word Embeddings and Neural Networks

**Questions covered:** Q1 (preprocessing and sentence vectors), Q2 (word analogies), and Q3 (single-hidden-layer neural networks).

This notebook uses the Amazon Reviews dataset and compares Google News Word2Vec, GloVe, and FastText embeddings.

## Before running
1. Download the Amazon Reviews dataset from [Kaggle](https://www.kaggle.com/datasets/kritanjalijain/amazon-reviews/data). Put `train.csv` and `test.csv` in `data/amazon_reviews/`.
2. Obtain the Google News Word2Vec, GloVe, and FastText vector files. Put them in `embeddings/` (or edit the paths in the configuration cell).
   - Word2Vec: Google News 300-dimensional vectors (`GoogleNews-vectors-negative300.bin.gz`).
   - GloVe: the downloaded GloVe text vector file.
   - FastText: `wiki-news-300d-1M-subword.vec` (or its unzipped version).
3. Install the dependencies in the next cell.

**Note:** These pretrained embedding files are large. Load one model at a time if RAM is limited. Google News Word2Vec can require several GB of RAM.

In [1]:
# Run once in a fresh environment
%pip install -q pandas numpy scikit-learn gensim nltk matplotlib seaborn tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, re, glob, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

RANDOM_STATE = 42
from pathlib import Path

cwd = Path.cwd()
candidate_dirs = [
    cwd / "amazon",
    cwd / "Assignment 10" / "amazon",
    cwd.parent / "Assignment 10" / "amazon",
]
DATA_DIR = next((path for path in candidate_dirs if (path / "train.csv").exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find amazon/train.csv or amazon/test.csv. "
        "Place the dataset files in Assignment 10/amazon/."
    )

EMBED_DIR = Path.cwd() / "embeddings"
if not EMBED_DIR.exists():
    EMBED_DIR = cwd / "Assignment 10" / "embeddings"
if not EMBED_DIR.exists():
    EMBED_DIR = cwd.parent / "Assignment 10" / "embeddings"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")

WORD2VEC_PATH = os.path.join(EMBED_DIR, "GoogleNews-vectors-negative300.bin.gz")
GLOVE_PATH = os.path.join(EMBED_DIR, "glove.6B.300d.txt")  # Change to your downloaded GloVe filename
FASTTEXT_PATH = os.path.join(EMBED_DIR, "wiki-news-300d-1M-subword.vec")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(EMBED_DIR, exist_ok=True)

print("Training CSV:", TRAIN_CSV)
print("Testing CSV:", TEST_CSV)

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training CSV: c:\Users\User\NLP\Assignment 10\amazon\train.csv
Testing CSV: c:\Users\User\NLP\Assignment 10\amazon\test.csv


## Q1 – Load, preprocess and sample the Amazon Reviews data

The code below detects common review and label column names automatically. If the dataset uses an unusual schema, set `REVIEW_COL` and `LABEL_COL` manually after inspecting the printed column names.

The requested sample sizes are **10,000 training reviews** and **1,000 test reviews**, selected with stratification so the class proportions are maintained.

In [3]:
# Amazon review CSV files are headerless: label, title, review text.
train_raw = pd.read_csv(TRAIN_CSV, header=None, names=["label", "title", "review"])
test_raw = pd.read_csv(TEST_CSV, header=None, names=["label", "title", "review"])

print("Train shape:", train_raw.shape)
print("Train columns:", train_raw.columns.tolist())
print("Test shape:", test_raw.shape)
print("Test columns:", test_raw.columns.tolist())
display(train_raw.head())

Train shape: (3599999, 3)
Train columns: ['2', 'Stuning even for the non-gamer', 'This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I have ever played it has the best music! It backs away from crude keyboarding and takes a fresher step with grate guitars and soulful orchestras. It would impress anyone who cares to listen! ^_^']
Test shape: (399999, 3)
Test columns: ['2', 'Great CD', 'My lovely Pat has one of the GREAT voices of her generation. I have listened to this CD for YEARS and I still LOVE IT. When I\'m in a good mood it makes me feel better. A bad mood just evaporates like sugar in the rain. This CD just oozes LIFE. Vocals are jusat STUUNNING and lyrics just kill. One of life\'s hidden gems. This is a desert isle CD in my book. Why she never made it big is just beyond me. Everytime I play this, no matter black, white, young, old, m

,2,Stuning even for the non-gamer,This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I have ever played it has the best music! It backs away from crude keyboarding and takes a fresher step with grate guitars and soulful orchestras. It would impress anyone who cares to listen! ^_^
0,2,The best soundtrack ever to anything.,I'm reading a lot of reviews saying that this ...
1,2,Amazing!,This soundtrack is my favorite music of all ti...
2,2,Excellent Soundtrack,I truly like this soundtrack and I enjoy video...
3,2,"Remember, Pull Your Jaw Off The Floor After He...","If you've played the game, you know how divine..."
4,2,an absolute masterpiece,I am quite sure any of you actually taking the...


In [5]:
def choose_column(df, candidates, description):
    lower_to_original = {str(c).lower(): c for c in df.columns}
    for name in candidates:
        if name in lower_to_original:
            return lower_to_original[name]
    raise ValueError(
        f"Could not identify {description} column. Available columns: {df.columns.tolist()}. "
        f"Set the column name manually."
    )

# Common schemas for Amazon review datasets
REVIEW_CANDIDATES = ["review", "content", "text", "review_text", "reviews", "body"]
LABEL_CANDIDATES = ["label", "sentiment", "class", "polarity", "rating", "score"]

REVIEW_COL = choose_column(train_raw, REVIEW_CANDIDATES, "review")
LABEL_COL = choose_column(train_raw, LABEL_CANDIDATES, "label")

print("Detected review column:", REVIEW_COL)
print("Detected label column:", LABEL_COL)

# Use the same column names in test data
if REVIEW_COL not in test_raw.columns or LABEL_COL not in test_raw.columns:
    raise ValueError("The detected review/label columns are not present in test.csv. Set them manually.")

train_df = train_raw[[REVIEW_COL, LABEL_COL]].dropna().copy()
test_df = test_raw[[REVIEW_COL, LABEL_COL]].dropna().copy()
train_df.columns = ["review", "label"]
test_df.columns = ["review", "label"]

# Stratified sampling: preserve the original class distribution
if len(train_df) < 10000 or len(test_df) < 1000:
    raise ValueError("The provided CSV files contain fewer rows than the required sample sizes.")

train_sample, _ = train_test_split(
    train_df, train_size=10000, stratify=train_df["label"], random_state=RANDOM_STATE
)
test_sample, _ = train_test_split(
    test_df, train_size=1000, stratify=test_df["label"], random_state=RANDOM_STATE
)

train_sample = train_sample.reset_index(drop=True)
test_sample = test_sample.reset_index(drop=True)

print("Sampled train:", train_sample.shape)
print("Sampled test:", test_sample.shape)
print("\nTraining class proportions:")
print(train_sample["label"].value_counts(normalize=True).sort_index())
print("\nTesting class proportions:")
print(test_sample["label"].value_counts(normalize=True).sort_index())

ValueError: Could not identify review column. Available columns: ['2', 'Stuning even for the non-gamer', 'This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I have ever played it has the best music! It backs away from crude keyboarding and takes a fresher step with grate guitars and soulful orchestras. It would impress anyone who cares to listen! ^_^']. Set the column name manually.

In [ ]:
# Required preprocessing:
# 1. lowercase
# 2. remove punctuation and numbers
# 3. tokenize
# 4. remove stopwords

STOP_WORDS = set(stopwords.words("english"))

def preprocess_review(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\\s]", " ", text)  # removes punctuation, digits and other symbols
    tokens = text.split()
    tokens = [word for word in tokens if word not in STOP_WORDS]
    return tokens

train_sample["tokens"] = train_sample["review"].apply(preprocess_review)
test_sample["tokens"] = test_sample["review"].apply(preprocess_review)

train_sample["clean_review"] = train_sample["tokens"].apply(lambda x: " ".join(x))
test_sample["clean_review"] = test_sample["tokens"].apply(lambda x: " ".join(x))

display(train_sample[["review", "tokens"]].head())

## Load the three pretrained embedding models

The loaders support:
- Word2Vec Google News binary format (`.bin` or `.bin.gz`).
- GloVe text format.
- FastText text vectors (`.vec`).

Gensim's `KeyedVectors` is used to provide a common interface (`key_to_index`, vector lookup and `most_similar`) for all three models.

In [ ]:
from gensim.models import KeyedVectors

def load_word2vec(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Word2Vec file not found: {path}")
    return KeyedVectors.load_word2vec_format(path, binary=True)

def load_glove(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"GloVe file not found: {path}")
    # GloVe is a text file without a word2vec header
    return KeyedVectors.load_word2vec_format(path, binary=False, no_header=True)

def load_fasttext(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"FastText file not found: {path}")
    # FastText .vec files normally have a header: vocab_size vector_size
    return KeyedVectors.load_word2vec_format(path, binary=False)

def load_all_embeddings():
    models = {}
    print("Loading Google News Word2Vec...")
    models["Word2Vec"] = load_word2vec(WORD2VEC_PATH)
    print("Loading GloVe...")
    models["GloVe"] = load_glove(GLOVE_PATH)
    print("Loading FastText...")
    models["FastText"] = load_fasttext(FASTTEXT_PATH)
    for name, model in models.items():
        print(name, "vocabulary:", len(model.key_to_index), "| dimension:", model.vector_size)
    return models

# Run after placing all three files in the embeddings directory.
embedding_models = load_all_embeddings()

### Sentence representation

For Q1, each review is represented by the **average of the pretrained vectors** of the words that occur in its vocabulary. Words missing from a model's vocabulary are skipped. If no token has a vector, the review receives a zero vector.

The function also supports TF-IDF-weighted averages, which are used in Q3.

In [ ]:
def sentence_vector(tokens, keyed_vectors, weights=None):
    vectors = []
    used_weights = []
    for token in tokens:
        if token in keyed_vectors:
            vectors.append(keyed_vectors[token])
            used_weights.append(1.0 if weights is None else weights.get(token, 0.0))

    if not vectors:
        return np.zeros(keyed_vectors.vector_size, dtype=np.float32)

    vectors = np.asarray(vectors, dtype=np.float32)
    used_weights = np.asarray(used_weights, dtype=np.float32)

    if weights is None:
        return vectors.mean(axis=0)

    total = used_weights.sum()
    if total <= 0:
        return vectors.mean(axis=0)
    return np.average(vectors, axis=0, weights=used_weights)

# Q1 representations: average vectors for train and test
q1_vectors = {}
for model_name, kv in embedding_models.items():
    print("Creating Q1 vectors for", model_name)
    X_train = np.vstack([sentence_vector(tokens, kv) for tokens in tqdm(train_sample["tokens"])])
    X_test = np.vstack([sentence_vector(tokens, kv) for tokens in tqdm(test_sample["tokens"])])
    q1_vectors[model_name] = (X_train, X_test)

    print(model_name, "train matrix:", X_train.shape, "test matrix:", X_test.shape)

# Save the sampled data and Q1 vectors for later use
train_sample.drop(columns=["tokens"]).to_csv("sampled_train.csv", index=False)
test_sample.drop(columns=["tokens"]).to_csv("sampled_test.csv", index=False)
for name, (Xtr, Xte) in q1_vectors.items():
    safe = name.lower()
    np.save(f"{safe}_q1_train_vectors.npy", Xtr)
    np.save(f"{safe}_q1_test_vectors.npy", Xte)

## Q2 – Word analogy task

For an analogy `A - B + C = ?`, the target vector is calculated as:

\[
v_{target} = v(A) - v(B) + v(C)
\]

The nearest word to this target is returned, excluding the three input words. The two examples in the assignment are tested first.

The full `questions-words.txt` file is then evaluated. Its lines beginning with `:` identify analogy sections; the first two sections (capitalization/grammatical categories) are skipped, following the common word-analogy evaluation convention. Only questions whose four words are present in a model's vocabulary are included in the accuracy denominator. The notebook reports both the number evaluated and coverage, so the accuracy is not misinterpreted.

In [ ]:
ANALOGY_URL = "https://raw.githubusercontent.com/nicholas-leonard/word2vec/master/questions-words.txt"
ANALOGY_FILE = "questions-words.txt"

if not os.path.exists(ANALOGY_FILE):
    urllib.request.urlretrieve(ANALOGY_URL, ANALOGY_FILE)

def solve_analogy(kv, a, b, c, topn=1):
    words = [a.lower(), b.lower(), c.lower()]
    missing = [w for w in words if w not in kv]
    if missing:
        return None
    results = kv.most_similar(
        positive=[words[0], words[2]],
        negative=[words[1]],
        topn=topn + 3
    )
    for word, score in results:
        if word.lower() not in set(words):
            return word, float(score)
    return None

examples = [
    ("good", "bad", "excellent"),
    ("delivery", "late", "fast")
]

example_rows = []
for model_name, kv in embedding_models.items():
    for a, b, c in examples:
        result = solve_analogy(kv, a, b, c)
        example_rows.append({
            "Model": model_name,
            "Analogy": f"{a} - {b} + {c}",
            "Predicted word": result[0] if result else "OOV / no result",
            "Similarity": result[1] if result else np.nan
        })

example_results = pd.DataFrame(example_rows)
display(example_results)

In [ ]:
def load_analogy_questions(path, skip_first_sections=2):
    questions = []
    section = ""
    section_index = -1
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip().lower()
            if not line:
                continue
            if line.startswith(":"):
                section_index += 1
                section = line[1:].strip()
                continue
            parts = line.split()
            if len(parts) == 4 and section_index >= skip_first_sections:
                questions.append((section, parts[0], parts[1], parts[2], parts[3]))
    return questions

analogy_questions = load_analogy_questions(ANALOGY_FILE)
print("Analogy questions after skipping first two sections:", len(analogy_questions))

def evaluate_analogies(kv, questions):
    correct = 0
    evaluated = 0
    skipped = 0
    by_section = {}

    for section, a, b, c, expected in tqdm(questions):
        if any(w not in kv for w in (a, b, c, expected)):
            skipped += 1
            continue
        result = solve_analogy(kv, a, b, c)
        if result is None:
            skipped += 1
            continue
        predicted = result[0].lower()
        evaluated += 1
        is_correct = predicted == expected
        correct += int(is_correct)
        if section not in by_section:
            by_section[section] = [0, 0]
        by_section[section][0] += int(is_correct)
        by_section[section][1] += 1

    accuracy = correct / evaluated if evaluated else 0.0
    coverage = evaluated / len(questions) if questions else 0.0
    section_rows = [
        {"Section": sec, "Correct": vals[0], "Evaluated": vals[1],
         "Accuracy": vals[0] / vals[1] if vals[1] else 0}
        for sec, vals in by_section.items()
    ]
    return accuracy, evaluated, skipped, coverage, pd.DataFrame(section_rows)

analogy_summary = []
analogy_section_results = {}
for model_name, kv in embedding_models.items():
    print("\nEvaluating", model_name)
    acc, evaluated, skipped, coverage, section_df = evaluate_analogies(kv, analogy_questions)
    analogy_summary.append({
        "Model": model_name,
        "Correct": round(acc * evaluated),
        "Evaluated": evaluated,
        "Skipped (OOV)": skipped,
        "Coverage": coverage,
        "Analogy accuracy": acc
    })
    analogy_section_results[model_name] = section_df

analogy_results = pd.DataFrame(analogy_summary)
display(analogy_results)

## Q3 – Single-hidden-layer neural network

For each embedding model, two sentence representations are compared:

**A. Simple average:** average of all available word vectors in a review.

**B. TF-IDF-weighted average:** each word vector is weighted by that word's TF-IDF score. TF-IDF is fitted on the training reviews only, then applied to the test reviews to avoid data leakage.

A one-hidden-layer `MLPClassifier` is trained for each representation and embedding model. Macro F1-score is the main comparison metric. The hidden layer has 128 neurons; the same settings and random seed are used for all six experiments.

In [ ]:
# Fit TF-IDF on training tokens only.
# Identity tokenizer means we use the already-preprocessed tokens.
tfidf = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    min_df=2
)

train_text_for_tfidf = train_sample["tokens"].apply(lambda x: " ".join(x))
test_text_for_tfidf = test_sample["tokens"].apply(lambda x: " ".join(x))

tfidf_train = tfidf.fit_transform(train_text_for_tfidf)
tfidf_test = tfidf.transform(test_text_for_tfidf)

vocabulary = tfidf.vocabulary_

def row_to_weight_dict(row, feature_names):
    indices = row.indices
    values = row.data
    return {feature_names[i]: float(v) for i, v in zip(indices, values)}

feature_names = tfidf.get_feature_names_out()
train_weight_dicts = [
    row_to_weight_dict(tfidf_train[i], feature_names) for i in range(tfidf_train.shape[0])
]
test_weight_dicts = [
    row_to_weight_dict(tfidf_test[i], feature_names) for i in range(tfidf_test.shape[0])
]

print("TF-IDF train matrix:", tfidf_train.shape)
print("TF-IDF test matrix:", tfidf_test.shape)

In [ ]:
y_train = train_sample["label"].to_numpy()
y_test = test_sample["label"].to_numpy()

def build_weighted_matrix(token_lists, weight_dicts, keyed_vectors):
    output = []
    for tokens, weights in tqdm(zip(token_lists, weight_dicts), total=len(token_lists)):
        output.append(sentence_vector(tokens, keyed_vectors, weights=weights))
    return np.vstack(output).astype(np.float32)

results = []
trained_models = {}

for model_name, kv in embedding_models.items():
    print("\n==========", model_name, "==========")

    # A: simple average vectors (already created for Q1)
    X_train_avg, X_test_avg = q1_vectors[model_name]

    # B: TF-IDF weighted average vectors
    X_train_tfidf = build_weighted_matrix(
        train_sample["tokens"].tolist(), train_weight_dicts, kv
    )
    X_test_tfidf = build_weighted_matrix(
        test_sample["tokens"].tolist(), test_weight_dicts, kv
    )

    representations = {
        "Average": (X_train_avg, X_test_avg),
        "TF-IDF weighted": (X_train_tfidf, X_test_tfidf)
    }

    for representation_name, (Xtr, Xte) in representations.items():
        print(f"Training {model_name} – {representation_name}")
        clf = MLPClassifier(
            hidden_layer_sizes=(128,),  # exactly one hidden layer
            activation="relu",
            solver="adam",
            alpha=1e-4,
            batch_size=64,
            learning_rate_init=1e-3,
            max_iter=30,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=5,
            random_state=RANDOM_STATE,
            verbose=False
        )
        clf.fit(Xtr, y_train)
        predictions = clf.predict(Xte)

        macro_f1 = f1_score(y_test, predictions, average="macro")
        accuracy = accuracy_score(y_test, predictions)

        results.append({
            "Embedding": model_name,
            "Representation": representation_name,
            "Accuracy": accuracy,
            "Macro F1": macro_f1,
            "Iterations": clf.n_iter_
        })
        trained_models[(model_name, representation_name)] = clf

        print("Accuracy:", round(accuracy, 4))
        print("Macro F1:", round(macro_f1, 4))
        print(classification_report(y_test, predictions, zero_division=0))

In [ ]:
results_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False)
display(results_df)

plt.figure(figsize=(10, 5))
for representation in results_df["Representation"].unique():
    subset = results_df[results_df["Representation"] == representation]
    plt.plot(subset["Embedding"], subset["Macro F1"], marker="o", label=representation)

plt.title("Macro F1-score comparison across embedding models")
plt.xlabel("Embedding model")
plt.ylabel("Macro F1-score")
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Detailed evaluation of the best Q3 model

The following cell prints the classification report and confusion matrix for the best-performing embedding/representation combination.

In [ ]:
best_row = results_df.iloc[0]
best_key = (best_row["Embedding"], best_row["Representation"])
best_model = trained_models[best_key]

if best_row["Representation"] == "Average":
    X_best_test = q1_vectors[best_row["Embedding"]][1]
else:
    best_kv = embedding_models[best_row["Embedding"]]
    X_best_test = build_weighted_matrix(
        test_sample["tokens"].tolist(), test_weight_dicts, best_kv
    )

best_predictions = best_model.predict(X_best_test)
print("Best configuration:", best_key)
print(classification_report(y_test, best_predictions, zero_division=0))

cm = confusion_matrix(y_test, best_predictions)
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest")
plt.title(f"Confusion Matrix – {best_key[0]} / {best_key[1]}")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.colorbar()
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.tight_layout()
plt.show()

## Save results

The tables below are the results to use in the assignment report. Run the notebook fully before copying values; actual scores depend on the exact embedding files, dataset version, preprocessing, and available vocabulary.

Do not invent or pre-fill numerical results. Report the measured outputs from your own run.

In [ ]:
analogy_results.to_csv("q2_analogy_results.csv", index=False)
results_df.to_csv("q3_macro_f1_results.csv", index=False)

print("Saved:")
print("- q2_analogy_results.csv")
print("- q3_macro_f1_results.csv")
display(analogy_results)
display(results_df)

## Suggested report discussion

- **Q1:** State the detected review/label columns, preprocessing steps, class distributions before/after stratified sampling, embedding dimensions, and the resulting matrix shapes (`10000 × d` and `1000 × d`).
- **Q2:** Include the two example analogy predictions and the full analogy accuracy, evaluated count, skipped/OOV count, and coverage for each embedding. Accuracy is calculated only over questions for which all four words are in the vocabulary.
- **Q3:** Include the six macro F1-scores (three embeddings × two sentence representations). Discuss whether TF-IDF weighting improves performance over the unweighted average for each embedding.
- **Reproducibility:** Mention the random seed, hidden layer size (128), activation (ReLU), optimizer (Adam), maximum epochs (30), and early stopping.

**Important interpretation:** The analogy benchmark contains many types of relations. The assignment's two illustrative analogies are not the complete evaluation set; the reported model accuracy comes from the analogy dataset.